In [1]:
# --- Bootstrap: run this first --------------------------------------------
# This notebook lives in drkrag/notebooks/. Resolve the project root (the folder that
# contains `src/`), put it on sys.path so `from src import ...` and `pdf_parallel_utils`
# resolve, make it the working directory so the on-disk Chroma store / caches / relative
# paths behave exactly as before, and load the project-root .env.
import os, sys
from pathlib import Path

def _find_project_root():
    """Locate the project root (folder containing src/embeddings.py) regardless of the
    notebook's working directory: search upward, then a bounded distance DOWNWARD — the
    latter handles editors (e.g. VS Code) that run notebooks from the PARENT workspace
    folder instead of the notebook's own directory."""
    here = Path.cwd().resolve()
    marker = Path("src") / "embeddings.py"
    for c in [here, *here.parents]:
        if (c / marker).exists():
            return c
    for hit in list(here.glob("*/" + marker.as_posix())) + list(here.glob("*/*/" + marker.as_posix())):
        return hit.parent.parent
    return here

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / '.env', override=True)  # override any stale/empty key already in the kernel env
print('PROJECT_ROOT:', PROJECT_ROOT, '| .env present:', (PROJECT_ROOT / '.env').exists())


PROJECT_ROOT: /Users/dhanyakrishnan/Documents/AI Workspace/drkrag | .env present: True


In [2]:
%load_ext autoreload
%autoreload 2


## Hybrid RAG over the neurology corpus — one local generator, one local judge

Everything in this notebook runs on this machine or on a free hosted tier. No paid API is used.

| Stage | Component | Where it runs |
|---|---|---|
| Chunk embedding | `BAAI/bge-m3` (sentence-transformers, 1024-dim) | local |
| Vector index | Pinecone serverless, cosine similarity | hosted, free Starter tier |
| Keyword index | BM25 via `bm25s`, over the same `chunks` list | local, rebuilt each kernel start |
| Fusion | Reciprocal Rank Fusion (`rrf_k=60`) | local |
| Generation | `qwen2.5:14b` on Ollama | local |
| Judging | `gemma4:12b` on Ollama, 4 metrics | local |

**Why two indexes.** The vector arm matches *meaning* — chunks near the query in embedding
space. The BM25 arm matches *literal terms*. Rare clinical tokens (tofersen, SOD1,
El Escorial, physostigmine) are exactly where embeddings blur and BM25 is exact, so `hybrid`
fuses both and lets a chunk found by both arms outrank one topping only a single arm.

**Why one generator and one judge.** Earlier versions compared Mistral and Gemini alongside
qwen. Both are quota-limited on the free tier and were the only source of failed runs, so
generation is now local-only. The judge is a *different* local model from the generator, which
avoids self-scoring bias while keeping the whole loop free and rate-limit-free.


In [3]:
# --- Ingest -> local embeddings -> Pinecone, plus a BM25 index over the same chunks ---
from langsmith import traceable
from src import (
    process_all_pdfs, split_documents, EmbeddingManager, PineconeVectorStore,
    BM25Index, content_key,
)
from src.pinecone_store import MAX_TOP_K

# 1. Load PDFs (cached after the first run) and chunk them.
docs   = process_all_pdfs(str(PROJECT_ROOT / "pdfs"), True)
# Chunking is sized for the retrieval band, not the model ceiling. BGE-M3 accepts 8192
# tokens but a single 1024-d vector averaging that much text loses the specific fact, so
# the useful band is ~256-512 tokens. 1500 chars (~375 tokens) keeps a management protocol
# or a diagnostic-criteria table whole — at the previous 1000/200 an ALS management list
# split across two chunks and neither half retrieved well. 20% overlap is unchanged.
# See notebooks/RAGApp_hybrid_Architecture.md for the four pressures that set this.
chunks = split_documents(docs, chunk_size=1500, chunk_overlap=300)

# 2. Embed the chunks LOCALLY, store the vectors in Pinecone.
# Embedding runs on this machine (sentence-transformers, no API cost); only the resulting
# vectors and their chunk text are sent to Pinecone. Free Starter tier restricts serverless
# indexes to AWS us-east-1, which is the default in PineconeVectorStore. Needs
# PINECONE_API_KEY in the project-root .env (already loaded by the bootstrap cell).

# --- Embedding model: BAAI/bge-m3 -------------------------------------------------
# Swapped from all-MiniLM-L6-v2 (22M params, 384-dim) to BGE-M3 (568M params, 1024-dim).
# BGE-M3 is trained for multilingual + long-context retrieval and is markedly stronger on
# domain vocabulary, which is the weakness MiniLM showed on this corpus (the smoke query
# `tofersen SOD1` topped out at cosine 0.284 — near-nothing in embedding space).
#
# Three consequences of the swap, all of which bite silently if missed:
#   * DIMENSION. 384 -> 1024. A Pinecone index is fixed to the dimension it was created
#     with, so this CANNOT reuse `drkrag-pdf-documents`; PineconeVectorStore raises rather
#     than corrupting it. A new index name is the fix, and the old 384-dim index is left
#     intact so the earlier experiments stay reproducible.
#   * NO PREFIXES. BGE-M3 is symmetric — unlike bge-large-en-v1.5 it needs no
#     "Represent this sentence..." instruction, and unlike Nemotron no "query:"/"passage:".
#     It is absent from _MODEL_PREFIXES in src/embeddings.py, which is correct: the lookup
#     defaults to "" and `is_query` becomes a no-op. Do not add prefixes for it.
#   * COST. ~2.3 GB of weights and ~25x MiniLM's compute. First run downloads the model;
#     embedding all 21,135 chunks takes tens of minutes on MPS rather than a couple, and
#     the loaded model then stays resident in this kernel for the rest of the session —
#     budget it alongside the 9 GB / 8 GB Ollama models (see the evaluation cell).
# encode_batch_size drops 32 -> 8: MiniLM's batch size would spike memory on a model 25x
# its size. Chunking is 1500/300 (~375-token chunks) — well inside BGE-M3's 8192-token
# window, and sized for the retrieval band rather than the model ceiling.
EMBED_MODEL = "BAAI/bge-m3"
EMBED_DIM   = 1024

embedding_manager = EmbeddingManager(model_name=EMBED_MODEL, encode_batch_size=8)
vectorstore       = PineconeVectorStore(                 # serverless index, cosine SIMILARITY
    index_name="drkrag-pdf-bge-m3",                      # separate from the 384-dim index
    dimension=EMBED_DIM,
)

# Fail fast if the model's real width ever disagrees with the index we just created —
# otherwise the mismatch surfaces as an opaque upsert error thousands of chunks later.
_actual = embedding_manager._embedding_dimension()
assert _actual == EMBED_DIM, f"{EMBED_MODEL} emits {_actual}-dim vectors, index expects {EMBED_DIM}"

# Skip the embed+upsert when the index already holds exactly this corpus. Ids are
# content hashes, so re-running would upsert byte-identical vectors — idempotent, but it
# costs a full BGE-M3 pass (~30 min for 21k chunks) for no change. Any edit to the PDFs
# or the chunk size changes the count and re-triggers the build.
_existing = vectorstore.count()
if _existing == len(chunks):
    print(f"Index already holds {_existing} vectors for these {len(chunks)} chunks — skipping embed.")
else:
    print(f"Index has {_existing} vectors for {len(chunks)} chunks — embedding.")
    texts      = [d.page_content for d in chunks]
    embeddings = embedding_manager.generate_embeddings(texts)   # document chunks -> "passage"
    vectorstore.add_documents(chunks, embeddings)

# 3. Build the BM25 keyword index over the SAME `chunks` list. Rebuilt each kernel start
# (~1s for 11k chunks) rather than persisted, so it can never drift out of sync with `chunks`.
bm25_index = BM25Index().build(chunks)


# ---------------------------------------------------------------------------
# 4. Retrieval: four modes over the same corpus.
#
#   mode="vector"          -> Pinecone cosine similarity only (semantic)
#   mode="keyword"         -> BM25 only                        (lexical / exact-term)
#   mode="hybrid"          -> both arms fused with RRF          (default)
#   mode="hybrid_weighted" -> both arms fused with normalised score blending
#
# @traceable logs each call as its own span in LangSmith (inputs=query+mode,
# outputs=hits), so retrieval shows up as a separate, inspectable step in the trace and you
# can see WHICH retriever produced the context for a given answer. Every hit keeps its
# metadata (source_file, page) so answers can still be cited.
# ---------------------------------------------------------------------------

def _vector_search(query: str, n: int):
    """Pinecone arm: embed the query, fetch the n nearest DISTINCT chunks.

    Deduplicated by content_key, with adaptive over-fetching, because the same text can
    legitimately exist under two ids (the id keys on source_file + page, so identical
    boilerplate in two textbooks is two records — provenance worth keeping). Widening
    the window until n DISTINCT chunks are found stops the generator from receiving six
    copies of one paragraph as if it were six pieces of evidence.

    Two Pinecone-specific details, both of which are silent failures if missed:
      * `similarity` is used AS RETURNED. Chroma reported a cosine DISTANCE and this
        function used to convert it with `1 - dist`; Pinecone returns cosine
        SIMILARITY, so applying that conversion again would invert the ranking while
        still looking like plausible numbers.
      * The fetch window is capped at MAX_TOP_K. Responses carrying metadata are
        limited to ~4 MB, so the x5 widening below cannot be allowed to ask for tens of
        thousands of matches.
    """
    q_vec = embedding_manager.generate_embeddings([query], is_query=True)[0]
    # Pinecone's stats are eventually consistent; a 0 here would just mean "not settled
    # yet", so fall back to the requested window rather than trusting it as an emptiness
    # signal. The cap is what actually bounds the loop.
    total = vectorstore.count() or (n * 3)
    ceiling = min(total, MAX_TOP_K)
    fetch = min(n * 3, ceiling)

    while True:
        raw = vectorstore.query(q_vec, n_results=fetch)

        hits, seen = [], set()
        for r in raw:
            key = content_key(r["content"])
            if key in seen:
                continue
            seen.add(key)
            hits.append({
                "content": r["content"],
                "metadata": r["metadata"],
                # Cosine similarity in 0..1 — higher is closer in embedding space.
                "similarity": r["similarity"],
                "rank": len(hits) + 1,
                "content_key": key,
            })
            if len(hits) == n:
                return hits

        if fetch >= ceiling:    # scanned as deep as we are allowed; this is all there is
            return hits
        fetch = min(fetch * 5, ceiling)


def _fuse_rrf(vector_hits, keyword_hits, rrf_k: int = 60):
    """Reciprocal Rank Fusion: score = sum over arms of 1 / (rrf_k + rank).

    Why rank and not raw score: cosine similarity is bounded 0..1 while BM25 is
    unbounded and corpus-relative, so the two are not on a comparable scale and any
    fixed weighting of the raw numbers shifts meaning from query to query. RRF only
    reads position, which both arms agree on. rrf_k=60 is the standard damping
    constant — it flattens the curve so rank 1 doesn't dominate ranks 2-5 outright,
    which is what lets a chunk found by BOTH arms outrank a chunk topping only one.
    """
    fused = {}
    for arm, hits in (("vector", vector_hits), ("keyword", keyword_hits)):
        for h in hits:
            key = h["content_key"]
            entry = fused.setdefault(key, {
                "content": h["content"], "metadata": h["metadata"],
                "similarity": None, "bm25_score": None,
                "vector_rank": None, "keyword_rank": None,
                "fusion_score": 0.0, "content_key": key,
            })
            entry["fusion_score"] += 1.0 / (rrf_k + h["rank"])
            if arm == "vector":
                entry["vector_rank"] = h["rank"]
                entry["similarity"]  = h["similarity"]
            else:
                entry["keyword_rank"] = h["rank"]
                entry["bm25_score"]   = h["bm25_score"]
    return sorted(fused.values(), key=lambda x: x["fusion_score"], reverse=True)


def _fuse_weighted(vector_hits, keyword_hits, alpha: float = 0.5):
    """Alternative fusion: alpha * norm(cosine) + (1-alpha) * norm(bm25).

    Min-max normalises each arm's scores WITHIN this query before blending, which is
    the only way to make the two scales addable. Provided so you can compare fusion
    strategies; RRF is the default because this normalisation is unstable when an arm
    returns few hits or near-identical scores (the min-max range collapses).
    alpha=1.0 is pure vector, alpha=0.0 is pure keyword.
    """
    def _norm(values):
        vals = [v for v in values if v is not None]
        if not vals:
            return {}
        lo, hi = min(vals), max(vals)
        span = (hi - lo) or 1.0
        return {"lo": lo, "span": span}

    v_n = _norm([h["similarity"] for h in vector_hits])
    k_n = _norm([h["bm25_score"] for h in keyword_hits])

    def _blank(h):
        return {
            "content": h["content"], "metadata": h["metadata"],
            "similarity": None, "bm25_score": None,
            "vector_rank": None, "keyword_rank": None,
            "fusion_score": 0.0, "content_key": h["content_key"],
        }

    fused = {}
    for h in vector_hits:
        entry = fused.setdefault(h["content_key"], _blank(h))
        entry["similarity"]  = h["similarity"]
        entry["vector_rank"] = h["rank"]
        if v_n and h["similarity"] is not None:
            entry["fusion_score"] += alpha * (h["similarity"] - v_n["lo"]) / v_n["span"]
    for h in keyword_hits:
        entry = fused.setdefault(h["content_key"], _blank(h))
        entry["bm25_score"]   = h["bm25_score"]
        entry["keyword_rank"] = h["rank"]
        if k_n:
            entry["fusion_score"] += (1 - alpha) * (h["bm25_score"] - k_n["lo"]) / k_n["span"]
    return sorted(fused.values(), key=lambda x: x["fusion_score"], reverse=True)



# ---------------------------------------------------------------------------
# Abstention floor — the pipeline's ability to say "not in these documents".
#
# Without this, retrieve() ALWAYS returns k chunks: Pinecone hands back the six nearest
# vectors however far away they are, BM25 its best lexical matches however weak. The
# generator is never shown that retrieval failed, so "I don't know" is the one thing its
# context never suggests — the refusal branch is unreachable, not merely unused.
#
# WHY NOT GATE ON THE FUSION SCORE. RRF is built from RANK alone: the top chunk scores
# ~1/61 + 1/61 = 0.0328 whether it is a perfect match or unrelated boilerplate. It carries
# no relevance information and cannot be thresholded. The floor must read the underlying
# arm scores instead.
#
# WHY *EITHER* ARM CLEARS IT (or, not and). BM25 exists here precisely to catch rare
# clinical tokens the embedding blurs — tofersen, SOD1, El Escorial. Requiring the vector
# arm to agree would abstain on exactly the queries BM25 was added for. So: abstain only
# when BOTH arms are weak.
#
# Thresholds are STARTING POINTS, not calibrated values. Run the calibration cell below
# on this corpus and set them from the observed gap before trusting the gate.
ABSTAIN_ENABLED = True
# Calibrated from the separation report below (BGE-M3, 1500/300 chunks):
#   cosine: in-corpus min 0.573 | out-of-corpus max 0.493  -> SEPARABLE, midpoint ~0.53
#   bm25:   in-corpus min 4.66  | out-of-corpus max 5.18   -> OVERLAPPING, unusable
MIN_COSINE = 0.53   # vector arm: bounded 0..1 and comparable across queries
MIN_BM25   = None   # lexical arm: EXCLUDED from the gate — see clears_floor()


def arm_scores(vector_hits, keyword_hits) -> tuple:
    """Best score from each arm: (cosine, bm25). Missing arm contributes 0.0."""
    best_cos  = max((h.get("similarity")  or 0.0) for h in vector_hits)  if vector_hits  else 0.0
    best_bm25 = max((h.get("bm25_score")  or 0.0) for h in keyword_hits) if keyword_hits else 0.0
    return best_cos, best_bm25


def clears_floor(vector_hits, keyword_hits) -> bool:
    """True when there is evidence good enough to answer from.

    Cosine ONLY. BM25 is deliberately excluded from this decision, though it remains a
    full partner in RANKING — the two roles need different properties:

      * Ranking only compares scores WITHIN one query, where BM25 is excellent.
      * A floor compares a score against a fixed constant ACROSS queries, which requires
        a bounded, query-comparable scale. Cosine over normalised embeddings has one;
        BM25 is unbounded and corpus-relative, so the same number means different things
        for different queries.

    The measured consequence: "How do I center a div in CSS?" scored bm25 5.18 — HIGHER
    than the genuine clinical question "Therapy for anticholinergic syndrome" at 4.66 —
    because words like "center" are common in medical prose. Under the previous
    `cos >= MIN_COSINE or bm25 >= MIN_BM25` rule, BM25 alone forced an answer even though
    cosine had correctly rejected it, and the bot answered a CSS question from neurology
    textbooks. No BM25 threshold can separate those two populations: their ranges cross.

    `or` was the specific defect — it let the unreliable arm overrule the reliable one.
    Switching to `and` would be no better: the noisy arm would then veto correct refusals
    and cause over-refusal instead.
    """
    best_cos, _best_bm25 = arm_scores(vector_hits, keyword_hits)
    return best_cos >= MIN_COSINE


@traceable
def retrieve(query: str, k: int = 6, mode: str = "hybrid",
             candidate_k: int = 20, alpha: float = 0.5, rrf_k: int = 60,
             abstain: bool = None):
    """Retrieve the top-k chunks for `query`.

    Args:
        query: the user question.
        k: how many chunks to hand to the generator.
        mode: "vector" | "keyword" | "hybrid" | "hybrid_weighted".
        candidate_k: how many chunks EACH arm fetches before fusion. Must exceed k —
            fusion can only reward agreement it can see, so a chunk ranked 15th by
            vector and 2nd by keyword is only rescued if both arms looked that deep.
        alpha: only for "hybrid_weighted" — vector's share of the blend.
        rrf_k: only for "hybrid" — RRF damping constant.
        abstain: apply the relevance floor. None follows ABSTAIN_ENABLED. When the floor
            is not cleared this returns an EMPTY LIST — callers must treat that as
            "nothing relevant found" and refuse, not as "no results this time".

    Returns a list of dicts, best first. Score fields are all present but may be None
    when an arm didn't return that chunk (e.g. bm25_score is None for a chunk only
    the vector arm found). See the legend printed below for how to read each one.
    """
    if mode not in ("vector", "keyword", "hybrid", "hybrid_weighted"):
        raise ValueError(
            f"Unknown mode {mode!r} — expected 'vector', 'keyword', 'hybrid' or 'hybrid_weighted'."
        )
    abstain = ABSTAIN_ENABLED if abstain is None else abstain

    # Fusion modes need candidate_k from each arm; single-arm modes need only k. When the
    # floor is active a single-arm mode still probes the OTHER arm with one hit, because
    # either arm clearing the floor is enough to proceed.
    n = candidate_k if mode in ("hybrid", "hybrid_weighted") else k
    vector_hits  = (_vector_search(query, n)     if mode != "keyword"
                    else (_vector_search(query, 1) if abstain else []))
    keyword_hits = (bm25_index.search(query, n)  if mode != "vector"
                    else (bm25_index.search(query, 1) if abstain else []))

    if abstain and not clears_floor(vector_hits, keyword_hits):
        return []       # the caller's cue to refuse rather than invent

    if mode == "vector":
        return vector_hits[:k]
    if mode == "keyword":
        return keyword_hits[:k]

    fused = (_fuse_rrf(vector_hits, keyword_hits, rrf_k=rrf_k) if mode == "hybrid"
             else _fuse_weighted(vector_hits, keyword_hits, alpha=alpha))
    return fused[:k]


# --- Smoke test: same query through all three modes, side by side ---
_q = "anticoagulation after stroke"
for _mode in ("vector", "keyword", "hybrid"):
    _hits = retrieve(_q, k=3, mode=_mode)
    print(f"\n=== mode={_mode} ===")
    for _h in _hits:
        _src  = _h["metadata"].get("source_file", "?")
        _page = _h["metadata"].get("page", "?")
        _bits = []
        if _h.get("similarity")   is not None: _bits.append(f"cos={_h['similarity']:.3f}")
        if _h.get("bm25_score")   is not None: _bits.append(f"bm25={_h['bm25_score']:.2f}")
        if _h.get("fusion_score") is not None: _bits.append(f"rrf={_h['fusion_score']:.4f}")
        print(f"  {' '.join(_bits):<42} {_src} p{_page}")

# i How to read the scores
#   cos   - cosine similarity, 0..1. How close the chunk is to the query in embedding
#           space. Comparable across queries. ~1.0 = near-identical meaning.
#   bm25  - BM25 relevance, unbounded and corpus-relative. How strongly the chunk matched
#           the query's RARE terms. Only comparable within one result list; a bm25 of 7 on
#           one query says nothing about a 7 on another.
#   rrf   - fusion score, typically 0.008-0.033 for two arms. Built from RANK, not
#           relevance, so its absolute value carries no meaning: use it only to order this
#           list. A chunk found by both arms scores roughly double one found by one arm at
#           the same rank - that agreement is the whole point of hybrid.
#   vector_rank / keyword_rank - the chunk's position in each arm, None if that arm missed
#           it. A hit with both filled is corroborated evidence.


Loaded 5314 pages from cache: /Users/dhanyakrishnan/Documents/AI Workspace/drkrag/data/pdf_documents_cache.pkl


Split 5314 documents into 21030 chunks

Example chunk:
Content:  Joseph Jankovic, MD
Professor of Neurology
Distinguished Chair in Movement Disorders
Director of Parkinson’s Disease Center and  
Movement Disorders Clinic
Department of Neurology
Baylor College of Me...
Metadata: {'producer': 'Foxit PhantomPDF Printer Version 3.1.0.0815', 'creator': 'Nitro Pro 13 (13.19.2.356)', 'creationdate': '2021-04-07T23:41:51+03:30', 'source': "/Users/dhanyakrishnan/Documents/AI Workspace/drkrag/pdfs/Bradley's Neurology in Clinical Practice 8th Edition.pdf", 'file_path': "/Users/dhanyakrishnan/Documents/AI Workspace/drkrag/pdfs/Bradley's Neurology in Clinical Practice 8th Edition.pdf", 'total_pages': 3078, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2021-07-23T23:08:10+06:00', 'trapped': '', 'modDate': "D:20210723230810+06'00'", 'creationDate': "D:20210407234151+03'30'", 'page': 1, 'source_file': "Bradley's Neurology in Clinical Practice 8th Edition.pd

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Model loaded successfully. Embedding dimension: 1024
Actual model device: mps:0, dtype: torch.float32


Pinecone index ready: drkrag-pdf-bge-m3


...

Generated embeddings with shape: (1, 1024)



=== mode=keyword ===
  bm25=5.74                                  Harrisons Neurology in Clinical Medicine, 3E.pdf p848
  bm25=5.60                                  Harrisons Neurology in Clinical Medicine, 3E.pdf p287
  bm25=5.50                                  Harrisons Neurology in Clinical Medicine, 3E.pdf p288
Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 1024)



=== mode=hybrid ===
  cos=0.653 bm25=5.50 rrf=0.0323             Harrisons Neurology in Clinical Medicine, 3E.pdf p288
  cos=0.639 bm25=5.60 rrf=0.0315             Harrisons Neurology in Clinical Medicine, 3E.pdf p287
  cos=0.630 bm25=5.35 rrf=0.0297             Bradley's Neurology in Clinical Practice 8th Edition.pdf p1219


### Calibrating the abstention floor


In [4]:
# --- Calibrate the abstention floor ------------------------------------------------
# MIN_COSINE and MIN_BM25 above are starting points, not measurements. This cell prints
# both arm scores for questions KNOWN to be in the corpus and questions known to be
# outside it. Set each threshold in the gap between the two groups; if the groups
# overlap, the floor cannot separate them and the thresholds are not trustworthy.
#
# Re-run this whenever the embedding model or the chunk size changes — both move the
# cosine distribution, and BM25 is corpus-relative by construction.
_in_corpus = [
    "What is the management of GBS if there is no improvement after giving IVIG?",
    "When to consider closing a PFO after stroke?",
    "tofersen SOD1 ALS",
    "Therapy for anticholinergic syndrome.",
]
_out_of_corpus = [
    "What is the capital of Peru?",
    "How do I center a div in CSS?",
    "What were the company's Q3 2025 revenues?",
    "Best sourdough starter hydration ratio?",
]

print(f"{'':<52} {'cosine':>8} {'bm25':>8}   verdict")
print("-" * 84)
_rows = []
for _label, _qs in (("IN ", _in_corpus), ("OUT", _out_of_corpus)):
    for _q in _qs:
        _v = _vector_search(_q, 1)
        _kw = bm25_index.search(_q, 1)
        _cos, _bm = arm_scores(_v, _kw)
        _pass = clears_floor(_v, _kw)
        _rows.append((_label, _cos, _bm))
        print(f"{_label} {_q[:48]:<48} {_cos:>8.3f} {_bm:>8.2f}   "
              f"{'answer' if _pass else 'ABSTAIN'}")

_in_cos  = [c for l, c, b in _rows if l == "IN "]
_out_cos = [c for l, c, b in _rows if l == "OUT"]
_in_bm   = [b for l, c, b in _rows if l == "IN "]
_out_bm  = [b for l, c, b in _rows if l == "OUT"]
print("\n--- separation ---")
print(f"  cosine: in-corpus min {min(_in_cos):.3f} | out-of-corpus max {max(_out_cos):.3f} "
      f"| {'SEPARABLE' if min(_in_cos) > max(_out_cos) else 'OVERLAPPING <-- floor unreliable'}")
print(f"  bm25:   in-corpus min {min(_in_bm):.2f} | out-of-corpus max {max(_out_bm):.2f} "
      f"| {'SEPARABLE' if min(_in_bm) > max(_out_bm) else 'OVERLAPPING <-- floor unreliable'}")
print(f"\n  current settings: MIN_COSINE={MIN_COSINE}  MIN_BM25={MIN_BM25}")
print("  Set each threshold midway between the two figures on its line above.")


                                                       cosine     bm25   verdict
------------------------------------------------------------------------------------
Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 1024)


IN  What is the management of GBS if there is no imp    0.674     9.25   answer
Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Generated embeddings with shape: (1, 1024)


OUT What were the company's Q3 2025 revenues?           0.387     4.89   ABSTAIN
Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 1024)


OUT Best sourdough starter hydration ratio?             0.499     4.35   ABSTAIN

--- separation ---
  cosine: in-corpus min 0.582 | out-of-corpus max 0.499 | SEPARABLE
  bm25:   in-corpus min 4.67 | out-of-corpus max 5.18 | OVERLAPPING <-- floor unreliable

  current settings: MIN_COSINE=0.53  MIN_BM25=None
  Set each threshold midway between the two figures on its line above.


## Generation

One local model, `qwen2.5:14b` on Ollama, wrapped in a traceable retrieve-then-generate bot.


In [5]:
# --- Generation: ONE local model ------------------------------------------------
# qwen2.5:14b runs on Ollama: no API key, no quota, no rate limits. Needs the model pulled
# (`ollama pull qwen2.5:14b`) and Ollama serving at http://localhost:11434.
#
# Mistral and Gemini were removed from this notebook. Both are quota-limited on their free
# tiers (Gemini returns 429 RESOURCE_EXHAUSTED partway through a full run) and were the only
# source of failed experiments, so generation is local-only now.
from langchain.chat_models import init_chat_model
from langsmith import traceable

GENERATOR = "qwen2.5:14b"
qwen_llm  = init_chat_model(GENERATOR, model_provider="ollama", temperature=0.0)

# --- Other models you could drop in (just swap the string) ---
# LangSmith traces any LangChain chat model regardless of provider; you just need its key.
# model = init_chat_model("anthropic:claude-opus-4-8")
# model = init_chat_model("openai:gpt-4o")
# model = init_chat_model("google_genai:gemini-2.0-flash")
# model = init_chat_model("phi4", model_provider="ollama", temperature=0.0)


# The exact wording the bot returns when retrieval finds nothing. Kept as a constant so
# the evaluator, the dataset's reference answers, and the bot all agree on one string.
REFUSAL = (
    "I could not find this in the source documents, so I am not answering. "
    "These sources are neurology reference texts; this question appears to fall outside them."
)


def make_rag_bot(llm, mode: str = "hybrid"):
    """Build a @traceable RAG bot bound to a generation LLM AND a retrieval mode.

    retrieve() is itself @traceable, so it appears as a nested span in each trace and
    LangSmith shows exactly which chunks the chosen retriever supplied for each answer.
    """
    @traceable
    def rag_bot(question: str) -> dict:
        hits = retrieve(question, k=6, mode=mode)

        # Empty hits means retrieval did not clear the floor. Refuse WITHOUT calling the
        # model: the refusal is then deterministic, costs no inference, and cannot be
        # talked out of by a persuasive-looking near-miss chunk. Sending an empty
        # "Documents:" block instead would read to the model as "answer from your own
        # knowledge", which is the hallucination this whole path exists to prevent.
        if not hits:
            return {"answer": REFUSAL, "documents": [],
                    "retrieval_mode": mode, "abstained": True}

        docs_string = " ".join(h["content"] for h in hits)
        instructions = f"""You are a helpful assistant who is good at analyzing source information and answering questions.       Use the following source documents to answer the user's questions.       If you don't know the answer, just say that you don't know.       Use three sentences maximum and keep the answer concise.

Documents:
{docs_string}"""
        ai_msg = llm.invoke([
            {"role": "system", "content": instructions},
            {"role": "user", "content": question},
        ])
        # retrieval_mode rides along in the output so it lands in the LangSmith trace and in
        # results.to_pandas(), making every row self-describing about how it was retrieved.
        return {"answer": ai_msg.text, "documents": hits,
                "retrieval_mode": mode, "abstained": False}
    return rag_bot


# One bot per retrieval mode, generation held FIXED at qwen2.5:14b. The retriever is
# therefore the ONLY variable between them.
rag_bots = {
    "vector":  make_rag_bot(qwen_llm, mode="vector"),
    "keyword": make_rag_bot(qwen_llm, mode="keyword"),
    "hybrid":  make_rag_bot(qwen_llm, mode="hybrid"),
}

# smoke test — both paths: one in-corpus question, one that must be refused
for _q in ("What are the guidelines for anticoagulation after a stroke?",
           "What is the capital of Peru?"):
    _out = rag_bots["hybrid"](_q)
    print(f"\nQ: {_q}\n   abstained={_out['abstained']}  docs={len(_out['documents'])}")
    print(f"   {_out['answer'][:160]}")


Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 1024)



Q: What are the guidelines for anticoagulation after a stroke?
   abstained=False  docs=6
   Anticoagulation is recommended long-term if atrial fibrillation persists after a stroke. For non-cardiogenic strokes caused by thromboembolism, data do not supp
Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 1024)



Q: What is the capital of Peru?
   abstained=True  docs=0
   I could not find this in the source documents, so I am not answering. These sources are neurology reference texts; this question appears to fall outside them.


## Evaluation dataset

14 questions: 10 Facharzt-level neurology questions with reference answers, plus 4 out-of-corpus questions that should be refused, pushed to LangSmith.
Creation is idempotent: re-running reuses the dataset instead of raising 409 Conflict.


In [6]:
from langsmith import Client

client=Client()

# Neurology evaluation set: 10 Facharzt-level questions with reference answers.
# (Expanded from 3 -> 10 so aggregate scores are statistically stable instead of swinging
#  33% on a single example.)
examples = [
    {
        "inputs": {"question": "What is the management of GBS if there is no improvement after giving IVIG?"},
        "outputs": {"answer": """GBS — no improvement after IVIG:
- Don't just repeat/escalate blindly. First reassess the diagnosis: acute-onset CIDP (A-CIDP), treatment-related fluctuation (TRF), or a mimic.
- A routine second IVIG course is not recommended — the SID-GBS trial showed no benefit and more adverse events in poor-prognosis patients.
- Do not combine IVIG + PLEX (no added benefit). Switching modalities isn't evidence-based either.
- TRF (deterioration after initial improvement, within ~2 months) → re-treat with IVIG. If ≥3 deteriorations or late/ongoing worsening → treat as A-CIDP (maintenance therapy).
- Optimize supportive care: serial vital capacity (intubate around <15–20 mL/kg or the 20/30/40 rule), autonomic monitoring, VTE prophylaxis, pain, early physio, ICU threshold low."""},
    },
    {
        "inputs": {"question": "Neurological conditions to think about when there is a sudden behavioral change?"},
        "outputs": {"answer": """Sudden behavioral change — neuro DDx:
NCSE / postictal state; stroke (right MCA, frontal, thalamic, caudate); HSV/limbic-autoimmune encephalitis (NMDA, LGI1); Wernicke; metabolic-toxic encephalopathy (hypoglycemia, hyponatremia, hepatic, uremic); drug intox/withdrawal, serotonin/anticholinergic/NMS; PRES; raised ICP/hydrocephalus; rapidly progressive dementia (CJD). Don't miss the reversible: hypoglycemia, Wernicke, NCSE, HSV, autoimmune encephalitis."""},
    },
    {
        "inputs": {"question": "When to consider closing a PFO after stroke?"},
        "outputs": {"answer": """PFO closure after stroke:
Consider closure when: cryptogenic embolic stroke (ESUS), patient roughly 18–60 years, thorough workup excludes other causes (esp. AF — prolonged rhythm monitoring), and high-risk PFO (large shunt and/or atrial septal aneurysm) or high RoPE score. Evidence: CLOSE, REDUCE, RESPECT, DEFENSE-PFO. Joint neuro–cardiology decision. Not indicated if a competing cause is found."""},
    },
    {
        "inputs": {"question": "How to differentiate different types of foot drop and wrist drop?"},
        "outputs": {"answer": """Foot drop vs wrist drop differentiation:

Foot drop — the key discriminator is inversion (tibialis posterior):
- Peroneal palsy: weak dorsiflexion + eversion; inversion preserved; sensory dorsum of foot/lateral leg; ankle jerk normal.
- L5 radiculopathy: dorsiflexion + eversion + inversion also weak; hip abduction (glut. medius) weak; L5 dermatome; ± back pain.
- Sciatic lesion: peroneal-predominant but tibial also affected (plantarflexion weak, ankle jerk lost). EMG of short head of biceps femoris separates sciatic from peroneal.
- Central/UMN: spastic, hyperreflexia.

Wrist drop — the key discriminator is triceps + sensory:
- Radial at spiral groove ("Saturday night"): wrist/finger extension weak, triceps spared, brachioradialis weak, sensory dorsal hand/first web.
- PIN (posterior interosseous): finger/thumb extension weak, wrist extends with radial deviation (ECRL spared), no sensory loss.
- High radial (axilla): triceps also weak.
- C7 radiculopathy: triceps weak + wider myotomal pattern, C7 sensory."""},
    },
    {
        "inputs": {"question": "Neurographic findings in neurogenic amyotrophy of shoulder."},
        "outputs": {"answer": """Neuralgic amyotrophy (Parsonage-Turner) — neurographic findings:
- Patchy, multifocal, non-contiguous involvement of individual plexus/nerve branches (long thoracic, suprascapular, axillary, anterior interosseous typical).
- Axonal pattern: reduced CMAP amplitude in affected nerves, conduction velocity normal; sensory NCS usually normal (motor-predominant, often proximal).
- EMG: acute — fibrillations/positive sharp waves; chronic — reinnervation (large polyphasic MUAPs, reduced recruitment) in the affected muscles."""},
    },
    {
        "inputs": {"question": "Vascular differential diagnoses for sudden loss of consciousness and what shouldn't be missed."},
        "outputs": {"answer": """Vascular DDx of sudden LOC — don't miss:
Cardiac syncope (arrhythmia/long QT, MI, severe AS, PE); SAH; basilar artery occlusion / top-of-basilar; aortic dissection; cardiac tamponade; ruptured AAA. Must not miss: SAH, basilar occlusion, PE, MI, aortic dissection, malignant arrhythmia. (Vasovagal/orthostatic are common but benign.)"""},
    },
    {
        "inputs": {"question": "Therapy principles of idiopathic Parkinson syndrome."},
        "outputs": {"answer": """Idiopathic Parkinson — therapy principles:
- Levodopa + DDC-inhibitor: most effective, favored in older patients.
- Dopamine agonists (pramipexole, ropinirole, rotigotine): younger patients to delay motor complications — watch impulse-control disorders.
- MAO-B inhibitors (rasagiline, safinamide) early/mild; COMT inhibitors (entacapone, opicapone) for wearing-off; amantadine for dyskinesia; anticholinergics for tremor (avoid in elderly).
- Advanced/device-aided: DBS (STN), levodopa-carbidopa intestinal gel, apomorphine pump.
- Start when functionally impairing; individualize by age/symptoms/comorbidity; add physio + non-motor symptom management."""},
    },
    {
        "inputs": {"question": "Therapy for anticholinergic syndrome."},
        "outputs": {"answer": """Anticholinergic syndrome — therapy:
- Physostigmine = antidote for severe central syndrome (crosses BBB). Contraindicated in TCA overdose (QRS widening → asystole risk); watch bradycardia/seizures.
- Supportive: benzodiazepines for agitation/seizures, active cooling for hyperthermia, IV fluids, cardiac monitoring, bladder catheter for retention, activated charcoal if early ingestion."""},
    },
    {
        "inputs": {"question": "Multiple sclerosis diagnostic and treatment guidelines."},
        "outputs": {"answer": """MS — diagnosis & treatment:
- Diagnosis: McDonald criteria (2017; 2024 revision adds optic nerve as a topography, central vein sign, kappa free light chains) — dissemination in space + time; CSF-specific oligoclonal bands can substitute for DIT; exclude mimics.
- Relapse: IV methylprednisolone 1 g × 3–5 d; plasma exchange if steroid-refractory.
- DMTs: moderate — IFN-β, glatiramer, teriflunomide, dimethyl fumarate; high-efficacy — anti-CD20 (ocrelizumab, ofatumumab), natalizumab, S1P modulators (fingolimod, siponimod, ozanimod), cladribine, alemtuzumab.
- PPMS: ocrelizumab. Increasing shift toward early highly-effective therapy. Plus symptomatic management."""},
    },
    {
        "inputs": {"question": "ALS diagnostic and management guidelines."},
        "outputs": {"answer": """ALS — diagnosis & management:
- Diagnosis: UMN + LMN signs with progression, exclusion of mimics. Criteria: El Escorial → Awaji → Gold Coast (2020): progressive motor impairment + UMN & LMN dysfunction in ≥1 region (or LMN in ≥2), other causes excluded. EMG: active + chronic denervation across regions.
- Disease-modifying: riluzole (survival ~+3 mo); edaravone (modest, jurisdiction-dependent); tofersen for SOD1-ALS.
- Multidisciplinary: NIV (improves survival/QoL), PEG/nutrition, secretion + spasticity + cramp management, communication aids, early palliative care and advance directives.
- Symptomatic: sialorrhea (amitriptyline, botulinum toxin), pseudobulbar affect, pain."""},
    },
]

# Every question above is answerable from these five textbooks. Stamp that explicitly, so
# the abstention evaluator has a label to compare the bot's behaviour against.
for _ex in examples:
    _ex["outputs"]["answerable"] = True

# --- Unanswerable questions: the test set for the refusal path ---------------------
# Without these the refusal path is untestable: 10 answerable questions can tell you the
# bot answers, never that it declines when it should. Each one is deliberately OUTSIDE a
# neurology corpus in a different way — a general-knowledge fact, a software question, a
# private business figure, an unrelated technical domain — so a single quirk of the floor
# cannot pass all four.
#
# The reference answer IS the refusal. `correctness` therefore rewards refusing and
# penalises a confident invention, which is exactly the behaviour under test.
examples += [
    {
        "inputs": {"question": "What is the capital of Peru?"},
        "outputs": {"answer": REFUSAL, "answerable": False},
    },
    {
        "inputs": {"question": "How do I center a div in CSS?"},
        "outputs": {"answer": REFUSAL, "answerable": False},
    },
    {
        "inputs": {"question": "What were the company's Q3 2025 revenues?"},
        "outputs": {"answer": REFUSAL, "answerable": False},
    },
    {
        "inputs": {"question": "What is the recommended torque for a bicycle crank bolt?"},
        "outputs": {"answer": REFUSAL, "answerable": False},
    },
]

print(f"dataset: {sum(e['outputs']['answerable'] for e in examples)} answerable + "
      f"{sum(not e['outputs']['answerable'] for e in examples)} unanswerable "
      f"= {len(examples)} examples")

### Create the dataset + examples in LangSmith (idempotent: reuse if it already exists).
# create_dataset() throws 409 Conflict if the name already exists, so re-running this cell
# used to crash. Guard with has_dataset(); only add examples on first creation.
# NEW NAME: this dataset now carries the four unanswerable questions and the `answerable`
# label. has_dataset() below reuses an existing dataset without adding examples, so keeping
# the old name would silently evaluate against the 10-question set and the abstention
# metric would have nothing to measure.
dataset_name = "Neurology RAG Eval + Refusal v1"
if client.has_dataset(dataset_name=dataset_name):
    dataset = client.read_dataset(dataset_name=dataset_name)
    print(f"Dataset '{dataset_name}' already exists ({dataset.example_count} examples) — reusing it.")
else:
    dataset = client.create_dataset(dataset_name=dataset_name)
    client.create_examples(dataset_id=dataset.id, examples=examples)
    print(f"Created dataset '{dataset_name}' with {len(examples)} examples.")


dataset: 10 answerable + 4 unanswerable = 14 examples


Dataset 'Neurology RAG Eval + Refusal v1' already exists (14 examples) — reusing it.


## The judge

One local judge model, `gemma4:12b`, scoring four boolean metrics:

- **correctness** — answer vs. the reference answer (the only metric using ground truth)
- **relevance** — does the answer address the question?
- **groundedness** — is the answer supported by the retrieved chunks? (hallucination check)
- **retrieval_relevance** — are the retrieved chunks related to the question at all?


In [7]:
from typing_extensions import Annotated,TypedDict,Literal
from langchain.chat_models import init_chat_model
import re

# --- Judge model: gemma4:12b, running LOCALLY on Ollama ---

# --- Why NOT llama3.1:8b (the previous judge) ---
# It produced systematically false-negative verdicts on correctness and retrieval_relevance.
# The cause was NOT bad parsing — parsing_error was None every time. It was constrained
# decoding interacting badly with a small model: method="json_schema" forces the fields in
# schema order (explanation, then verdict), so llama3.1:8b wrote ONE throat-clearing sentence
# into `explanation` ("To determine the relevance... I will analyze each fact individually."),
# the grammar closed the string, and it had to emit a true/false token having reasoned about
# nothing. Forced to answer cold it defaulted to false — deterministically, 3/3 at temperature 0.
# The giveaway was self-contradiction: runs scored groundedness=1.0 (the answer IS supported by
# these chunks) AND retrieval_relevance=0.0 (the chunks are unrelated to the question) on the
# same chunks. gemma4:12b has the headroom to actually reason inside the `explanation` field
# before committing to a verdict, which is what the explanation-before-answer ordering assumes.
#
#   method="json_schema" -> Ollama constrained decoding forces a clean true/false token.
#   include_raw=True      -> .invoke() returns {raw, parsed, ...} instead of raising, so a
#                            malformed judge response degrades gracefully (see _grade()).
JUDGE_MODEL = "gemma4:12b"
model = init_chat_model(JUDGE_MODEL, model_provider="ollama", temperature=0)

def _to_bool(v) -> bool:
    return str(v).strip().strip('.').strip().lower() in ("true", "yes", "1", "y", "t")

def _extract_bool(text: str, key: str) -> bool:
    """Fallback verdict parser: pull a true/false out of the judge's raw text when structured
    parsing fails, so a malformed judge response degrades gracefully instead of crashing."""
    if not text:
        return False
    m = re.search(rf'"?{key}"?\s*[:=]\s*"?(true|false|yes|no)"?', text, re.I)
    if m:
        return _to_bool(m.group(1))
    hits = re.findall(r'\b(true|false)\b', text, re.I)
    return _to_bool(hits[-1]) if hits else False

def _grade(grader, messages, key: str) -> bool:
    """Invoke a structured-output grader built with include_raw=True. That makes .invoke()
    return {"raw", "parsed", "parsing_error"} instead of raising on a bad parse — we use the
    parsed verdict when present, else fall back to scanning the raw text. This is the fix for
    the OutputParserException that previously produced None scores.

    NOTE: this fallback only catches PARSE failures. It cannot catch a judge that parses
    cleanly but reasons badly — that was the llama3.1:8b failure described above, and the
    only fix for it is a more capable judge model."""
    res = grader.invoke(messages)
    parsed = res.get("parsed") if isinstance(res, dict) else res
    if parsed:
        return _to_bool(parsed[key])
    raw = res.get("raw") if isinstance(res, dict) else None
    raw_text = getattr(raw, "text", "") or (getattr(raw, "content", "") if raw else "")
    return _extract_bool(str(raw_text), key)

# Correctness: Response vs reference answer
# Grade output schema
class CorrectnessGrade(TypedDict):
    # Note that the order in the fields are defined is the order in which the model will generate them.
    # It is useful to put explanations before responses because it forces the model to think through
    # its final response before generating it:
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    correct: Annotated[Literal['true','false'], ..., "True if the answer is correct, False otherwise."]

## correctness prompt

correctness_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION, the GROUND TRUTH (correct) ANSWER, and the STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Grade the student answers based ONLY on their factual accuracy relative to the ground truth answer. 
(2) Ensure that the student answer does not contain any conflicting statements.
(3) It is OK if the student answer contains more information than the ground truth answer, as long as it is factually accurate relative to the  ground truth answer.

Correctness:
A correctness value of True means that the student's answer meets all of the criteria.
A correctness value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""


# --- Shared abstention rule for the four answer-quality metrics -----------------------
# When the bot abstains there is no answer to grade, so these metrics would otherwise
# score a REFUSAL as if it were a bad answer — which is how a correct refusal came to be
# recorded as groundedness 0.25 / relevance 0.50 / retrieval_relevance 0.00.
#
# Instead, score the DECISION on those rows:
#   * unanswerable question + abstained -> True   (correctly declined; nothing to grade)
#   * answerable   question + abstained -> False  (over-refusal; the floor is too high)
# Answered rows fall through and are graded normally by the LLM judge.
#
# Returning a verdict rather than None keeps every metric defined on all 14 rows, so the
# means stay comparable across experiments. Note the consequence: on refusal rows all four
# metrics agree by construction, so they carry no independent signal there — they read as
# "did the system do the right thing", not "how good was the answer text".
def _abstention_verdict(outputs: dict, reference_outputs: dict):
    """Return True/False for an abstained row, or None if the bot actually answered."""
    if not bool(outputs.get("abstained", False)):
        return None
    return not bool(reference_outputs.get("answerable", True))


grader_llm=model.with_structured_output(CorrectnessGrade, method="json_schema", include_raw=True)
## evaluator
def correctness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for RAG answer accuracy"""
    _v = _abstention_verdict(outputs, reference_outputs)
    if _v is not None:
        return _v          # a correct refusal IS the correct output for this question
    answers = f"""\
QUESTION: {inputs['question']}
GROUND TRUTH ANSWER: {reference_outputs['answer']}
STUDENT ANSWER: {outputs['answer']}"""

    # Run evaluator
    return _grade(grader_llm, [
        {"role": "system", "content": correctness_instructions}, 
        {"role": "user", "content": answers}
    ], "correct")

# Relevance: Response vs input
# The flow is similar to above, but we simply look at the inputs and outputs without needing the reference_outputs. 
# Without a reference answer we can't grade accuracy, but can still grade relevance—as in, did the model address the user's question or not.
# Grade output schema
class RelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[Literal['true','false'], ..., "Provide the score on whether the answer addresses the question"]

# Grade prompt
relevance_instructions="""You are a teacher grading a quiz. 

You will be given a QUESTION and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is concise and relevant to the QUESTION
(2) Ensure the STUDENT ANSWER helps to answer the QUESTION

Relevance:
A relevance value of True means that the student's answer meets all of the criteria.
A relevance value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
relevance_llm = model.with_structured_output(RelevanceGrade, method="json_schema", include_raw=True)

# Evaluator
def relevance(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """A simple evaluator for RAG answer helpfulness."""
    _v = _abstention_verdict(outputs, reference_outputs)
    if _v is not None:
        return _v          # declining an unanswerable question IS the helpful response
    answer = f"QUESTION: {inputs['question']}\nSTUDENT ANSWER: {outputs['answer']}"
    return _grade(relevance_llm, [
        {"role": "system", "content": relevance_instructions}, 
        {"role": "user", "content": answer}
    ], "relevant")


# Groundedness: Response vs retrieved docs
# Another useful way to evaluate responses without needing reference answers is to check if the response is justified by (or "grounded in") the retrieved documents.
# Grade output schema
class GroundedGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    grounded: Annotated[Literal['true','false'], ..., "Provide the score on if the answer hallucinates from the documents"]

# Grade prompt
grounded_instructions = """You are a teacher grading a quiz. 

You will be given FACTS and a STUDENT ANSWER. 

Here is the grade criteria to follow:
(1) Ensure the STUDENT ANSWER is grounded in the FACTS. 
(2) Ensure the STUDENT ANSWER does not contain "hallucinated" information outside the scope of the FACTS.

Grounded:
A grounded value of True means that the student's answer meets all of the criteria.
A grounded value of False means that the student's answer does not meet all of the criteria.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM 
grounded_llm = model.with_structured_output(GroundedGrade, method="json_schema", include_raw=True)

# Evaluator
def groundedness(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """A simple evaluator for RAG answer groundedness."""
    # An abstention asserts nothing, so it cannot be ungrounded — but an over-refusal on
    # an answerable question is still a failure, so score the decision rather than
    # returning True unconditionally.
    _v = _abstention_verdict(outputs, reference_outputs)
    if _v is not None:
        return _v
    doc_string = "\n\n".join(doc["content"] for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nSTUDENT ANSWER: {outputs['answer']}"
    return _grade(grounded_llm, [
        {"role": "system", "content": grounded_instructions},
        {"role": "user", "content": answer}
    ], "grounded")


# Retrieval Relevance: Retrieved docs vs input
# Grade output schema
class RetrievalRelevanceGrade(TypedDict):
    explanation: Annotated[str, ..., "Explain your reasoning for the score"]
    relevant: Annotated[Literal['true','false'], ..., "True if the retrieved documents are relevant to the question, False otherwise"]

# Grade prompt
retrieval_relevance_instructions = """You are a teacher grading a quiz. 

You will be given a QUESTION and a set of FACTS provided by the student. 

Here is the grade criteria to follow:
(1) You goal is to identify FACTS that are completely unrelated to the QUESTION
(2) If the facts contain ANY keywords or semantic meaning related to the question, consider them relevant
(3) It is OK if the facts have SOME information that is unrelated to the question as long as (2) is met

Relevance:
A relevance value of True means that the FACTS contain ANY keywords or semantic meaning related to the QUESTION and are therefore relevant.
A relevance value of False means that the FACTS are completely unrelated to the QUESTION.

Explain your reasoning in a step-by-step manner to ensure your reasoning and conclusion are correct. 

Avoid simply stating the correct answer at the outset."""

# Grader LLM
retrieval_relevance_llm = model.with_structured_output(RetrievalRelevanceGrade, method="json_schema", include_raw=True)

def retrieval_relevance(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """An evaluator for document relevance"""
    # Abstention means nothing cleared the floor. On an unanswerable question that is the
    # CORRECT retrieval outcome, so it scores True; on an answerable one it is a miss.
    _v = _abstention_verdict(outputs, reference_outputs)
    if _v is not None:
        return _v
    doc_string = "\n\n".join(doc["content"] for doc in outputs["documents"])
    answer = f"FACTS: {doc_string}\nQUESTION: {inputs['question']}"

    # Run evaluator
    return _grade(retrieval_relevance_llm, [
        {"role": "system", "content": retrieval_relevance_instructions}, 
        {"role": "user", "content": answer}
    ], "relevant")


# Abstention: did the bot refuse exactly when it should have?
# -----------------------------------------------------------------------------------
# No LLM involved — this is a label comparison, so it is deterministic, free, and cannot
# be swayed by a persuasive wrong answer. It is the ONLY metric that scores the refusal
# path, and it reads in both directions:
#   * unanswerable question + abstained    -> True   (correctly declined)
#   * unanswerable question + answered     -> False  (hallucinated; the failure that matters)
#   * answerable question   + answered     -> True   (correctly proceeded)
#   * answerable question   + abstained    -> False  (over-refusal; the floor is too high)
# That last row is why the metric is not simply "refusal rate": a floor set high enough to
# refuse everything would score perfectly on a refusal-only measure.
def abstention(inputs: dict, outputs: dict, reference_outputs: dict) -> bool:
    """Did the bot's decision to answer-or-refuse match what the question warranted?"""
    answerable = bool(reference_outputs.get("answerable", True))
    abstained  = bool(outputs.get("abstained", False))
    return abstained != answerable


# The five metrics this single judge scores. Defined here, beside the evaluators
# themselves, so the evaluation cell below has no hidden ordering dependency.
evaluators = [correctness, groundedness, relevance, retrieval_relevance, abstention]
print("judge:", f"{JUDGE_MODEL} (local ollama)", "| evaluators:", [e.__name__ for e in evaluators])


judge: gemma4:12b (local ollama) | evaluators: ['correctness', 'groundedness', 'relevance', 'retrieval_relevance', 'abstention']


## Run the evaluation


In [8]:
# --- Evaluation, run in TWO PHASES ---
#
# Why two phases instead of the obvious single loop:
# qwen2.5:14b (9 GB) + gemma4:12b (8 GB) = 17 GB of weights on a 19.3 GB machine. They
# cannot both stay resident. A single interleaved loop alternates generate -> judge x4 ->
# generate on every example, so Ollama evicts and reloads an 8-9 GB model HUNDREDS of times;
# each reload takes tens of seconds from disk and eventually blows llama-server's startup
# timeout ("timed out waiting for llama-server to start", HTTP 500). A previous interleaved
# run spent 15.5 hours, finished 1 of 3 modes, and produced only 7 of 10 answers with ZERO
# scores - both generation and judging were failing.
#
# Splitting the work means only ONE model is needed at a time:
#   Phase 1 - all generations (qwen2.5:14b resident, NO evaluators attached)
#   Phase 2 - all judging via evaluate_existing (gemma4:12b resident)
# That is 2 model loads instead of ~300.
#
# It also makes judging cheap to retry: phase 1 results persist in LangSmith, so a judge
# failure or a change of judge model never costs you the generations again.
import json
import urllib.request

import pandas as pd
from IPython.display import display
from langsmith.evaluation import evaluate_existing

# Which retrieval modes to evaluate. Default is the one this notebook is about.
# Set to ["vector", "keyword", "hybrid"] for the full retriever comparison - same generator,
# same judge, so any score delta is attributable to retrieval alone. Costs 3x the runtime.
RETRIEVAL_MODES = ["hybrid"]

MODE_DETAIL = {
    "vector":  "pinecone cosine (BAAI/bge-m3, 1024-d)",
    "keyword": "bm25s lucene k1=1.5 b=0.75",
    "hybrid":  "RRF fusion (rrf_k=60, candidate_k=20)",
}
JUDGE = f"{JUDGE_MODEL} (local ollama)"


def unload_ollama_model(name: str):
    """Evict a model from Ollama's memory by requesting it with keep_alive=0.

    Belt-and-braces before switching phases: without this, Ollama may still be holding the
    generator when the first judge call arrives, which is the exact condition that caused
    the 15.5-hour failure. Best effort - a failure here is not fatal.
    """
    try:
        req = urllib.request.Request(
            "http://localhost:11434/api/generate",
            data=json.dumps({"model": name, "keep_alive": 0}).encode(),
            headers={"Content-Type": "application/json"},
        )
        urllib.request.urlopen(req, timeout=60).read()
        print(f"unloaded {name} from Ollama memory")
    except Exception as e:
        print(f"could not unload {name} ({e}) - continuing anyway")


# ============================ PHASE 1: GENERATION ============================
# No `evaluators` argument -> LangSmith records answers only. Judge model never loads.
print("=" * 70)
print(f"PHASE 1 - generating answers for {len(RETRIEVAL_MODES)} retrieval mode(s) "
      f"(generator: {GENERATOR}, no judging yet)")
print("=" * 70)

experiment_names = {}
generation_health = {}
for _mode in RETRIEVAL_MODES:
    _bot, _desc = rag_bots[_mode], MODE_DETAIL[_mode]

    # Bind the bot via a default arg: a bare closure over the loop variable would leave all
    # targets pointing at the LAST bot, silently running the same experiment N times.
    def _target(inputs: dict, _bot=_bot) -> dict:
        return _bot(inputs["question"])

    print(f"\n### [phase 1] generating: {_mode} ({_desc})")
    _res = client.evaluate(
        _target,
        data=dataset_name,
        # Judge name is in the prefix so these runs sort separately from any earlier sweep
        # judged by a different model, whose scores are not comparable to these.
        experiment_prefix=f"retr-{_mode}-gemma4judge",
        max_concurrency=1,
        metadata={
            "generation": f"{GENERATOR} (local ollama)",
            "retrieval": _mode,
            "retrieval_detail": _desc,
            "judges": JUDGE,
            "phase": "generation",
            "abstain": ABSTAIN_ENABLED,
            "min_cosine": MIN_COSINE,
            "min_bm25": MIN_BM25,
        },
    )
    experiment_names[_mode] = _res.experiment_name

    # Count how many rows actually carry an answer. A run whose generation failed has empty
    # outputs; the evaluators would then raise on outputs["answer"] and silently record nan.
    # Catching that HERE means a bad phase 1 costs seconds, not an hour of wasted judging.
    _df = _res.to_pandas()
    _ok = int(_df["outputs.answer"].notna().sum()) if "outputs.answer" in _df.columns else 0
    generation_health[_mode] = (_ok, len(_df))
    print(f"    -> experiment: {_res.experiment_name}  |  answers: {_ok}/{len(_df)}")

print("\n--- phase 1 integrity ---")
_incomplete = {m: v for m, v in generation_health.items() if v[0] != v[1] or v[1] == 0}
for _mode, (_ok, _tot) in generation_health.items():
    print(f"  {_mode:8} {_ok}/{_tot} answers {'OK' if _ok == _tot and _tot else '<-- INCOMPLETE'}")
if _incomplete:
    raise RuntimeError(
        f"Phase 1 incomplete for {list(_incomplete)}. Judging now would produce nan scores. "
        f"Check that Ollama is up and {GENERATOR} loads, then re-run this cell."
    )

# ============================= PHASE 2: JUDGING ==============================
# Free the generator before the judge loads, so the two never contend for RAM.
print("\n" + "=" * 70)
print(f"PHASE 2 - judging the stored runs (judge: {JUDGE_MODEL})")
print("=" * 70)
unload_ollama_model(GENERATOR)

eval_results = {}
for _mode, _exp_name in experiment_names.items():
    print(f"\n### [phase 2] judging: {_mode} ({_exp_name})")
    eval_results[_mode] = evaluate_existing(
        _exp_name,
        evaluators=evaluators,
        client=client,
        max_concurrency=0,      # sequential: one judge call at a time, one model resident
    )

# --- Score summary, SPLIT by whether the question was answerable ---------------------
# i Metrics are the fraction of questions the local gemma4:12b judge marked True, except
#   `abstention`, which is a deterministic label comparison and involves no judge.
#     correctness         — does the answer match the reference? (uses ground truth)
#                           on unanswerable rows the reference IS the refusal, so this
#                           rewards declining and penalises a confident invention
#     groundedness        — is every claim supported by the retrieved chunks?
#                           forced True on an abstention: a refusal asserts nothing
#     relevance           — does the answer address the question?
#                           forced False on an abstention — expected on unanswerable rows
#     retrieval_relevance — are the retrieved chunks related to the question?
#                           forced False on an abstention — the truthful reading
#     abstention          — did the answer-or-refuse decision match what the question
#                           warranted? Scores BOTH failure directions: hallucinating on an
#                           unanswerable question, and over-refusing an answerable one.
#
#   WHY THE SPLIT. Three of the five metrics are defined to be False on a correct
#   abstention, so pooling all 14 rows into one mean makes good refusal behaviour look
#   like a regression. Read the answerable block for answer quality and the unanswerable
#   block for refusal behaviour; `abstention` is the only metric meaningful in both.
#
#   RESOLUTION. 10 answerable rows -> steps of 0.1; 4 unanswerable rows -> steps of 0.25.
#   A single flipped question moves the unanswerable block a QUARTER of its range, so
#   treat that block as directional evidence, not a measurement.
def _summarise(df, score_cols):
    return {c.replace("feedback.", ""): df[c].mean() for c in score_cols}

summaries = {"answerable": {}, "unanswerable": {}, "all": {}}
for _mode, _res in eval_results.items():
    _df = _res.to_pandas()
    _score_cols = [c for c in _df.columns if c.startswith("feedback.")]

    # to_pandas() prefixes reference fields differently across langsmith versions, so
    # match on the suffix rather than guessing at "reference." vs "reference_outputs.".
    _ans_col = next((c for c in _df.columns if c.endswith("answerable")), None)
    summaries["all"][_mode] = _summarise(_df, _score_cols)
    if _ans_col is None:
        print(f"  [{_mode}] no `answerable` column found — reporting the pooled mean only. "
              f"Columns: {list(_df.columns)[:8]}")
        continue
    _mask = _df[_ans_col].astype("boolean").fillna(True)
    summaries["answerable"][_mode]   = _summarise(_df[_mask],  _score_cols)
    summaries["unanswerable"][_mode] = _summarise(_df[~_mask], _score_cols)

for _block, _label, _note in (
    ("answerable",   "ANSWERABLE questions (10)",   "answer quality"),
    ("unanswerable", "UNANSWERABLE questions (4)",  "refusal behaviour"),
):
    if not summaries[_block]:
        continue
    _sdf = pd.DataFrame(summaries[_block]).T
    _sdf.index.name = "retrieval_mode"
    print(f"\n=== {_label} — {_note} | generator: {GENERATOR} | judge: {JUDGE} ===")
    display(_sdf)

summary_df = pd.DataFrame(summaries["all"]).T
summary_df.index.name = "retrieval_mode"
print(f"\n=== POOLED across all 14 rows (see the split above before reading this) ===")
display(summary_df)

# Per-example detail, in case an aggregate hides the rare-term questions BM25 was added
# for (ALS/tofersen/SOD1, El Escorial, physostigmine) or a single over-refusal.
for _mode, _res in eval_results.items():
    print(f"\n=== retrieval mode: {_mode} ===")
    display(_res.to_pandas())


PHASE 1 - generating answers for 1 retrieval mode(s) (generator: qwen2.5:14b, no judging yet)

### [phase 1] generating: hybrid (RRF fusion (rrf_k=60, candidate_k=20))


View the evaluation results for experiment: 'retr-hybrid-gemma4judge-a60653ef' at:
https://smith.langchain.com/o/bd470e00-1cf2-461b-a459-0e215d334857/datasets/d3216886-0719-4183-827c-5af742d82d1f/compare?selectedSessions=c100ea2e-ad90-45eb-824f-3bbea5653701




0it [00:00, ?it/s]

Generating embeddings for 1 texts (as query)...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

,correctness,groundedness,relevance,retrieval_relevance,abstention
retrieval_mode,,,,,
hybrid,1.0,1.0,1.0,1.0,1.0



=== POOLED across all 14 rows (see the split above before reading this) ===


,correctness,groundedness,relevance,retrieval_relevance,abstention
retrieval_mode,,,,,
hybrid,0.692308,0.75,0.923077,0.833333,1.0



=== retrieval mode: hybrid ===


,inputs.question,error,reference.answer,reference.answerable,feedback.correctness,feedback.groundedness,feedback.relevance,feedback.retrieval_relevance,feedback.abstention,execution_time,example_id,id,outputs.abstained,outputs.answer,outputs.documents,outputs.retrieval_mode
0,When to consider closing a PFO after stroke?,None,PFO closure after stroke:\nConsider closure wh...,True,None,None,None,None,True,NaN,f254d8b3-5193-4ea6-8b85-bf21bfda3cf0,01a02e46-8cca-7850-bdd6-c9ae9f7796bc,NaN,NaN,NaN,NaN
1,What is the recommended torque for a bicycle c...,None,"I could not find this in the source documents,...",False,True,True,True,True,True,13.757202,e7e2aa4d-5a80-483d-b05d-d59655d10926,01a02e46-56f7-7812-84c4-cad8d7b5caf3,True,"I could not find this in the source documents,...",[],hybrid
2,Neurographic findings in neurogenic amyotrophy...,None,Neuralgic amyotrophy (Parsonage-Turner) — neur...,True,False,False,False,False,True,26.980863,d1b3ac2a-9b96-4bef-a3a3-0ae855e1b006,01a02e45-ed75-7ad2-80c1-752c9e39502c,False,Sensory nerve conduction studies are used to a...,"[{'bm25_score': 4.560629844665527, 'content': ...",hybrid
3,Therapy principles of idiopathic Parkinson syn...,None,Idiopathic Parkinson — therapy principles:\n- ...,True,True,True,True,True,True,23.226537,ba3a5e0f-e689-4c66-aa62-5084f6e89dda,01a02e45-92b6-7322-b51d-5ef78dc3ae08,False,The primary therapy for idiopathic Parkinson's...,"[{'bm25_score': 5.659379959106445, 'content': ...",hybrid
4,Therapy for anticholinergic syndrome.,None,Anticholinergic syndrome — therapy:\n- Physost...,True,True,True,True,True,True,23.281711,b2cf4b05-d6e8-4639-b916-025b88b37818,01a02e45-37c1-7750-bd47-c8c587f0c28e,False,Anticholinergic toxicity is treated with suppo...,"[{'bm25_score': 4.246479034423828, 'content': ...",hybrid
5,How to differentiate different types of foot d...,None,Foot drop vs wrist drop differentiation:\n\nFo...,True,False,False,True,True,True,19.463116,adfa024e-373f-4551-9dbf-a3cd4099d965,01a02e44-ebb7-7542-810f-beaa00697c00,False,Differentiating between types of foot drop inv...,"[{'bm25_score': 8.486396789550781, 'content': ...",hybrid
6,Neurological conditions to think about when th...,None,Sudden behavioral change — neuro DDx:\nNCSE / ...,True,False,True,True,True,True,20.363193,a7a39d8e-5009-4eb9-b416-ba43675e8e2b,01a02e44-9c28-7b81-a750-a01ee163373a,False,"When there's a sudden behavioral change, consi...","[{'bm25_score': 6.1348185539245605, 'content':...",hybrid
7,Vascular differential diagnoses for sudden los...,None,Vascular DDx of sudden LOC — don't miss:\nCard...,True,True,False,True,True,True,18.297273,9e32bd94-8565-4711-8951-999d1d602644,01a02e44-54ab-7e43-a157-0d4f459fff4c,False,"For sudden loss of consciousness, vascular cau...","[{'bm25_score': 7.731175422668457, 'content': ...",hybrid
8,How do I center a div in CSS?,None,"I could not find this in the source documents,...",False,True,True,True,True,True,9.630715,96ca22fb-392f-4ced-b537-54d27c21bf80,01a02e44-2f04-77c3-84eb-2a8e7c1418fe,True,"I could not find this in the source documents,...",[],hybrid
9,What is the management of GBS if there is no i...,None,GBS — no improvement after IVIG:\n- Don't just...,True,True,True,True,True,True,19.517351,84f7d292-23fc-408b-8e6f-fe87f84a0a74,01a02e43-e2c3-78f3-880e-c75de295888f,False,If there is no noticeable improvement followin...,"[{'bm25_score': 8.348459243774414, 'content': ...",hybrid
